## Initialization

In [ ]:
# Imports
# import pickle
# from pathlib import Path
from typing import Callable, Literal
from random import sample

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.signal import find_peaks
from scipy.interpolate import CubicSpline
import pandas as pd
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve
from data_processing.types import NasaGenerationSettings
from data_processing import helpers

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

## Experiment ID Input

In [ ]:
experiment_ids = input_experiment_ids()

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
calib_input = get_input_with_default(
    "Do you want to use new calibration? [y/n, or press Enter for yes]",
    "y",
    str
)

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
settings = get_nasa_generation_settings(calib_key)
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Pulse Height Distribution

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    exp_data[ExperimentDataKey.PSD_REPORT] = unclassified_df

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    signals_df = exp_data["signals_df"]
    signals_df = signals_df.astype("int32")
    print(signals_df.head())

    signals_np = signals_df.to_numpy()
    baselines = signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    signals_np = -signals_np + baselines
    signals_df = pd.DataFrame(signals_np, index=signals_df.index, columns=signals_df.columns)
    psd_report["peak_height"] = signals_df.max(axis=1)
    print(psd_report["peak_height"].max())
    print(psd_report["peak_height"].min())
    
    exp_data["signals_df"] = signals_df
    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    peak_height = psd_report["peak_height"]
    # print(neutron_energies.max())
    # print(neutron_energies.min())

    energy_bins = np.arange(0, 15000, step=200)
    
    Z_n, *_ = np.histogram(peak_height, bins=energy_bins)
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
        "all": {"standard": Z_n, "bins": energy_bins},
    }

In [ ]:
# moving average
for exp_id, exp_data in experiment_neutron_data.items():
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_histogram = phd_histogram_data["all"]["standard"]
    # phd_n_histogram = phd_histogram_data["neutron"]["standard"]
    # phd_g_histogram = phd_histogram_data["gamma"]["standard"]

    phd_moving_average = moving_average_centered(phd_histogram)
    # phd_n_moving_average = moving_average_centered(phd_n_histogram)
    # phd_g_moving_average = moving_average_centered(phd_g_histogram)

    phd_histogram_data["all"]["moving_average"] = phd_moving_average
    # phd_histogram_data["neutron"]["moving_average"] = phd_n_moving_average
    # phd_histogram_data["gamma"]["moving_average"] = phd_g_moving_average
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = phd_histogram_data

In [ ]:
# for exp_id, exp_data in experiment_neutron_data.items():
#     # energy_bins = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
#     psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
#     phd_histogram_data = exp_data[
#         ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["all"]
#     signals_df = exp_data["signals_df"]
#     energy_bins = phd_histogram_data["bins"]

#     energy_bin_mids = (energy_bins[1:] + energy_bins[:-1]) / 2
#     trace_bin_lo = energy_bins[::5][1:]
#     trace_bin_hi = energy_bins[1::5][1:]
#     trace_bins = list(zip(trace_bin_lo, trace_bin_hi))
#     traces = []
#     for trace_bin in trace_bins:
#         bin_lo, bin_hi = trace_bin
#         bin_mid = (bin_lo + bin_hi) / 2
#         matching_signals = psd_report[psd_report["peak_height"].between(bin_lo, bin_hi)]
#         if len(matching_signals) == 0:
#             continue
#         # TODO search for traces without dual peaks
#         for signal_id in matching_signals.index:
#             matching_trace = signals_df.loc[signal_id]
            
#             peaks, peak_data = find_peaks(matching_trace, height=200, prominence=50)
#             filtered_peaks = [peak for peak in peaks if abs(peak - 50) > 15]
#             if len(filtered_peaks) == 0:
#                 traces.append((bin_mid, matching_trace))
#                 break
#     traces = sample(traces, len(traces))

#     exp_data["selected_traces"] = traces

## Display

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["all"]
    # selected_traces = exp_data["selected_traces"]
    energy_bins = phd_histogram_data["bins"]
    phd_n_histogram = phd_histogram_data["standard"]
    phd_n_moving_avg = phd_histogram_data["moving_average"]
    phd_n_moving_avg = np.nan_to_num(phd_n_moving_avg)

    energy_bin_mids = (energy_bins[1:] + energy_bins[:-1]) / 2

    phd_spline = CubicSpline(energy_bin_mids, phd_n_moving_avg)
    phd_spline_x = np.linspace(0, energy_bins.max(), num=1000)
    phd_spline_y = phd_spline(phd_spline_x)

    fig, axs = plt.subplots(figsize=(12, 8))
    # fig.subplots_adjust(wspace=0)
    # ax1, ax2 = axs
    ax1 = axs

    # ax1.plot(phd_spline_y, phd_spline_x, color="black")
    # ax1.plot(phd_n_moving_avg, energy_bin_mids, color=bg_blue, linewidth=3)
    ax1.fill_betweenx(energy_bin_mids, 0, phd_n_moving_avg, color=bg_grey)
    # for i, (bin_mid, trace) in enumerate(selected_traces):
    #     trace_x = [x + i * 25 for x in range(len(trace))]
    #     ax2.plot(trace_x, trace, label=bin_mid)

    # ax1.xaxis.set_inverted(True)
    ax1.xaxis.set_ticks_position("top")
    ax1.xaxis.set_label_position("top")
    ax1.yaxis.set_inverted(True)
    # ax1.xaxis.set_tick_params(labelbottom=False)
    # ax1.yaxis.set_tick_params(labelbottom=False, bottom=False)
    # ax2.xaxis.set_tick_params(labelbottom=False)
    # ax2.yaxis.set_tick_params(labelbottom=False, direction="in")
    
    # ax1.spines["top"].set_visible(False)
    # ax1.spines["left"].set_visible(False)
    # ax2.spines["top"].set_visible(False)
    # ax2.spines["right"].set_visible(False)

    ax1.set_xlabel("Counts (x1000)", fontsize=fontsize)
    ax1.set_ylabel("Pulse height (ADC channel x1000)", fontsize=fontsize)
    ax1.xaxis.set_major_formatter(lambda x, pos: str(x // 1000))
    ax1.yaxis.set_major_formatter(lambda x, pos: str(x // 1000))
    ax1.tick_params(labelsize=fontsize)
    # ax2.set_xlabel("Time (ns)", fontsize=fontsize)

    # limits = (0, 5000)
    # ax1.set_ylim(*limits)
    # ax2.set_ylim(*limits)
    ax1.set_xlim(0, None)
    # ax2.set_xlim(None, 200)

    # for i, (bin_mid, trace) in enumerate(selected_traces):
    #     bin_mid_idx = np.where(energy_bin_mids == bin_mid)[0]
    #     bin_lo = float(energy_bins[bin_mid_idx][0])
    #     bin_hi = float(energy_bins[bin_mid_idx+1][0])
    #     bin_phd_x = np.linspace(bin_lo, bin_hi).reshape(-1, 1)
    #     bin_phd_y = phd_spline(bin_phd_x).reshape(-1, 1)
    #     bin_phd_xy = np.concatenate((bin_phd_y, bin_phd_x), axis=1)
    #     trace_max_x = float(trace.idxmax()) + (i * 25)
    #     bin_trace_xy = [[trace_max_x, bin_hi], [trace_max_x, bin_lo]]
        
    #     ax1_to_display = ax1.transData.transform
    #     ax2_to_display = ax2.transData.transform
    #     display_to_figure = fig.transFigure.inverted().transform

    #     bin_phd_xy = display_to_figure(ax1_to_display(bin_phd_xy))
    #     bin_trace_xy = display_to_figure(ax2_to_display(bin_trace_xy))
    #     bin_xy = np.concatenate((bin_phd_xy, bin_trace_xy))
        
    #     poly = mpl.patches.Polygon(bin_xy, closed=True, color="lightblue", alpha=0.3)
    #     fig.add_artist(poly)

    # get bin midpoints
    # make subplots (2 cols, 1 row, merged y-axis, no borders)
    # plot PHD histogram on left subplot (rotated)
    # pick 5 evenly spaced bins
    # for each bin, find first trace within that bin
    # (alternate, scan bin for trace with energy closest to midpoint of bin)
    # plot 5 traces on right subplot
    # make connecting lines between left subplot (bin midpoint, bin count) and right subplot (bin midpoint, trace height)

In [ ]:
input("Processing done, hit Enter to finish")
stop()